# Spark : évite les calculs en double avec `cache()`

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/MonShotData/blob/main/Optimisation%20du%20temps%20d%27exe%CC%81cution/Spark-Cache-DataFrame.ipynb)

📖 **Le shot complet** : [Pandas vers Spark : l'erreur qui ralentit tout](https://www.monshotdata.com/p/spark-cache-dataframe-lazy-evaluation)

---

Spark est paresseux (*lazy evaluation*) : une transformation ne calcule rien tant qu'aucune action ne le demande. Le revers, c'est que **chaque action rejoue toute la chaîne** : lecture du fichier comprise. Avec deux actions sur le même DataFrame, Spark lit ton CSV deux fois. Ici, tu vois le problème dans le plan d'exécution, puis tu le corriges avec `cache()`.

**À la fin de ce notebook, tu sauras :**

- distinguer les transformations (paresseuses) des actions (qui déclenchent le calcul) ;
- repérer dans le plan d'exécution (`explain()`) qu'une action relit le fichier source ;
- mettre un DataFrame en cache avec `cache()`, vérifier que Spark lit désormais le cache, puis libérer la place avec `unpersist()`.

### Avant de lancer

- **Installation** : la première cellule installe **PySpark**. Spark a besoin de Java, déjà présent sur Colab ; en local, installe un JDK (version 17 ou plus) si tu n'en as pas.
- **Les données** : un petit fichier `data.csv` de 100 000 lignes, généré dans le notebook (graine fixée) et écrit dans le dossier courant.
- **Spark en local** : la session tourne sur la machine du notebook (`local[*]`) : pas besoin de cluster pour comprendre le principe, qui est le même sur un vrai cluster.

🧪 *Testé le 22 novembre 2026 : exécution complète de haut en bas (Python 3.13, PySpark 4.3, Java 21, pandas 3.0). Les sorties affichées viennent de cette exécution.*

## 1. Mise en place

In [1]:
%pip install -q pyspark

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("shot-349-cache")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")  # moins de messages dans les sorties

On crée un fichier `data.csv` pour avoir quelque chose à lire. La colonne `colA` contient des entiers entre 0 et 20 : on gardera ensuite les lignes où `colA > 10`.

In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)  # graine fixée : mêmes données à chaque exécution
pd.DataFrame({
    "colA": rng.integers(0, 21, size=100_000),
    "colB": rng.normal(size=100_000).round(3),
}).to_csv("data.csv", index=False)

## 2. Transformations et actions

Spark distingue deux types d'opérations :

- les **transformations** (`filter`, `select`, `withColumn`...) créent un nouveau DataFrame à partir d'un autre, sans rien calculer ;
- les **actions** (`count`, `show`, `collect`, l'écriture d'un fichier...) déclenchent l'exécution de toute la chaîne de transformations.

On lit le CSV (DataFrame **A**), puis on applique une transformation (DataFrame **B**). On donne le schéma des colonnes à la lecture : avec `inferSchema=True`, Spark parcourrait le fichier dès cette ligne pour deviner les types, et la lecture ne serait plus vraiment paresseuse.

In [4]:
df_A = spark.read.csv("data.csv", header=True, schema="colA INT, colB DOUBLE")

# Applique une transformation
df_B = df_A.filter(df_A['colA'] > 10)

Ces deux lignes s'exécutent instantanément : rien n'a encore été lu. Spark a seulement noté **quoi faire**.

## 3. Le piège : deux actions, deux fois le même calcul

On réalise maintenant deux actions sur `df_B` : compter les lignes, puis les afficher.

**1re action** : Spark lit le CSV et applique le `filter`.

In [5]:
# Réalise la 1re action : compte les lignes de df_B
df_B.count()

47640

**2e action** : Spark **relit** le CSV et réapplique le même `filter`, car rien n'a été gardé de la première fois.

In [6]:
# Réalise la 2e action : affiche df_B
df_B.show(5)

+----+------+
|colA|  colB|
+----+------+
|  16|-0.853|
|  13| 0.459|
|  18| 0.544|
|  14|-0.541|
|  11|-0.154|
+----+------+
only showing top 5 rows


Pour le vérifier, regarde le plan d'exécution avec `explain()` : il part de la lecture du fichier (`FileScan csv`). C'est ce plan complet que Spark rejoue à chaque action.

In [7]:
df_B.explain()
print("df_B est en cache :", df_B.is_cached)

== Physical Plan ==
*(1) Filter (isnotnull(colA#0) AND (colA#0 > 10))
+- FileScan csv [colA#0,colB#1] Batched: false, DataFilters: [isnotnull(colA#0), (colA#0 > 10)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/data.csv], PartitionFilters: [], PushedFilters: [IsNotNull(colA), GreaterThan(colA,10)], ReadSchema: struct<colA:int,colB:double>


df_B est en cache : False


Sur 100 000 lignes, la double lecture passe inaperçue. Sur un gros dataset, ou avec une longue chaîne de transformations (jointures, agrégations), chaque action supplémentaire refait tout ce travail : c'est là que le programme ralentit.

## 4. La solution : `cache()`

`df.cache()` garde le résultat des transformations pour le réutiliser. On l'appelle juste après la transformation.

Attention : `cache()` est lui aussi paresseux. Il ne calcule rien tout de suite : le résultat est stocké lors de la **première action**.

In [8]:
df_A = spark.read.csv("data.csv", header=True, schema="colA INT, colB DOUBLE")

# Applique une transformation, puis met le résultat en cache
df_B = df_A.filter(df_A['colA'] > 10).cache()

In [9]:
# Réalise la 1re action : compte les lignes de df_B (le résultat est mis en cache)
df_B.count()

47640

In [10]:
# Réalise la 2e action : réutilise le résultat en cache
df_B.show(5)

+----+------+
|colA|  colB|
+----+------+
|  16|-0.853|
|  13| 0.459|
|  18| 0.544|
|  14|-0.541|
|  11|-0.154|
+----+------+
only showing top 5 rows


Le plan d'exécution change : Spark lit maintenant le résultat en cache (`InMemoryTableScan`) au lieu de repartir du CSV.

In [11]:
df_B.explain()
print("df_B est en cache :", df_B.is_cached)

== Physical Plan ==
InMemoryTableScan [colA#19, colB#20]
   +- InMemoryRelation [colA#19, colB#20], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Filter (isnotnull(colA#19) AND (colA#19 > 10))
            +- FileScan csv [colA#19,colB#20] Batched: false, DataFilters: [isnotnull(colA#19), (colA#19 > 10)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/data.csv], PartitionFilters: [], PushedFilters: [IsNotNull(colA), GreaterThan(colA,10)], ReadSchema: struct<colA:int,colB:double>


df_B est en cache : True


Où est stocké ce cache ? Regarde le niveau de stockage : pour un DataFrame, `cache()` utilise la mémoire **et** le disque. Les partitions qui ne tiennent pas en mémoire sont écrites sur le disque plutôt que recalculées.

In [12]:
print(df_B.storageLevel)

Disk Memory Deserialized 1x Replicated


Il existe aussi `df.persist()`, qui laisse choisir ce niveau de stockage (mémoire seule, disque seul...). Ici, `cache()` suffit.

## 5. Libérer la place avec `unpersist()`

Le cache occupe de la mémoire (et parfois du disque) tant que la session tourne. Dès que tu n'as plus besoin du DataFrame, libère-le avec `df.unpersist()`.

In [13]:
# Effectue des opérations
...

# Libère le DataFrame du cache
df_B.unpersist()
print("df_B est en cache :", df_B.is_cached)

df_B est en cache : False


In [14]:
spark.stop()  # arrête la session Spark à la fin du notebook

## 6. Ce qu'il faut retenir

- Les transformations Spark sont paresseuses : elles ne calculent rien tant qu'aucune action (`count`, `show`, `collect`, écriture...) ne le demande.
- Chaque action rejoue toute la chaîne de transformations, lecture du fichier comprise : deux actions sur le même DataFrame, c'est deux fois le même calcul.
- `explain()` montre le plan : `FileScan` sans cache, `InMemoryTableScan` avec cache.
- `df.cache()` garde le résultat (en mémoire, et sur le disque si besoin) ; il est rempli à la première action.
- Ne mets en cache que les DataFrames réutilisés plusieurs fois, et libère-les avec `df.unpersist()` dès que tu n'en as plus besoin.

---

## Pour aller plus loin

- [Tu optimises ton code, mais pas tes fichiers ? Grosse erreur.](https://www.monshotdata.com/p/optimisation-format-stockage) — lire un CSV coûte cher : compare les formats de stockage
- [Accélérer de 5x l'I/O des fichiers Parquet de Pandas](https://www.monshotdata.com/p/accelerer-io-parquet-de-pandas) — Parquet, le format que Spark lit le mieux
- [Tous les notebooks des shots](https://github.com/moncoachdata/MonShotData) — le code Python de la newsletter, rangé par thème